# PDGL Identification Notebook (Zhang et al. 2022 Scheme)

This notebook implements a **Potentially Dangerous Glacial Lakes (PDGL)** identification workflow
based on the quantitative hazard scheme of **Zhang et al. (2022)**.

For a given glacial lake inventory (e.g. HKH 2025), the notebook:

1. Loads **lake polygons**, **glacier outlines**, and an **SRTM (or similar) DEM mosaic**.
2. Computes a **slope raster** from the DEM.
3. Derives five hazard factors for each lake:
   - G11: Mean slope of parent glacier (degrees)
   - R1: Area of steep (>30°) non-glacierized terrain around lake (km²)
   - D3: Mean slope of the moraine dam (approximated) (degrees)
   - H7: Watershed area upstream of lake (km²) *(optional, advanced)*
   - L9: Lake perimeter (km)
4. Normalizes factors to [0, 1] and combines them with Zhang et al. weights:
   - Weights: 0.27 (G11), 0.25 (R1), 0.21 (D3), 0.17 (H7), 0.10 (L9)
5. Computes a **hazard index** and classifies it into 5 classes using **Jenks natural breaks**:
   - very_low, low, medium, high, very_high
6. Flags **PDGLs** as lakes in the **high** and **very_high** classes.

⚠️ Notes:
- This workflow can be run separately for your 2020 and 2025 lake inventories.
- The H7 (watershed area) factor is implemented as an **optional advanced section** using `pysheds`.
  If you skip it, the hazard index will still be computed (H7 treated as NaN and normalized accordingly).

Fill in the configuration paths and run the notebook top-to-bottom.

In [ ]:
# Optional installs (uncomment and run once if needed)
# %pip install geopandas rasterio shapely mapclassify pysheds

In [14]:
# === Imports ===
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.features import geometry_mask

from shapely.geometry import mapping, box
from shapely.ops import unary_union
from shapely.validation import make_valid

import mapclassify

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)

print('Imports OK')

Imports OK


In [15]:
# === Configuration ===

# Paths to your data (update these)
LAKES_PATH    = "D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/inputs/hkh_lakes_2025.zip"      # or .shp
GLACIERS_PATH = "D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/inputs/glims_download_19805.zip"       # RGI or other glacier outlines
DEM_PATH      = "D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/outputs/HKH_SRTM_30m_mosaic.tif"  # DEM mosaic covering your lakes
DEM_PATH_PROJ      = "D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/outputs/HKH_SRTM_30m_mosaic_proj.tif"
DEM_PATH_PROJ_SUBSET      = "D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/outputs/HKH_SRTM_30m_mosaic_proj_subset.tif"
# Projected CRS for area/slope calculations (metre-based projection)
AREA_CRS = "EPSG:6933"  # World Cylindrical Equal Area; change to regional EA if preferred

# Basic attribute names
AREA_COL = "area_km2"
ELEV_COL = "elev_mean"   # Optional: if you already have lake elevation

# Output PDGL-annotated lakes
OUT_PDGL_PATH = "lakes_2025_PDGL.gpkg"

# Slope raster (will be created from DEM)
SLOPE_PATH = "D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/outputs/HKH_SRTM_30m_slope_deg.tif"

# Parameters for factors
R1_BUFFER_M = 2000   # buffer radius around lakes for R1 (2 km)
R1_SLOPE_THRESH = 30.0  # degrees

# Optional: projected DEM path for watershed computations (H7 via pysheds)
#PROJECTED_DEM = "D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/outputs/HKH_SRTM_30m_proj.tif"

print('Config set. Update paths above before running.')

Config set. Update paths above before running.


In [16]:
# === Load lakes and glaciers, clean geometries ===

lakes = gpd.read_file(LAKES_PATH)
glaciers = gpd.read_file(GLACIERS_PATH)

print('Original CRS - lakes:', lakes.crs)
print('Original CRS - glaciers:', glaciers.crs)

if lakes.crs is None or glaciers.crs is None:
    raise ValueError('Lakes or glaciers have no CRS defined. Please set CRS before proceeding.')

# Reproject to AREA_CRS
lakes = lakes.to_crs(AREA_CRS)
glaciers = glaciers.to_crs(AREA_CRS)

# Fix invalid geometries
lakes['geometry'] = lakes.geometry.apply(make_valid)
glaciers['geometry'] = glaciers.geometry.apply(make_valid)

# Basic attributes
lakes[AREA_COL] = lakes.geometry.area / 1e6  # m² -> km²
lakes['perimeter_km'] = lakes.geometry.length / 1000.0

if 'lake_id' not in lakes.columns:
    lakes['lake_id'] = [f'lake_{i}' for i in range(len(lakes))]

print('Lakes loaded:', len(lakes))
print('Glaciers loaded:', len(glaciers))
lakes.head()

Original CRS - lakes: EPSG:4326
Original CRS - glaciers: EPSG:4326
Lakes loaded: 15353
Glaciers loaded: 872552


,source_img,area_m2,lake_id,geometry,area_km2,perimeter_km
0,20250902_060832_78_24da_3B_Visual_clip_reproje...,1322.085138,1,"POLYGON ((7480317.435 4235339.235, 7480319.604...",0.001321,0.133774
1,20250902_060832_78_24da_3B_Visual_clip_reproje...,733.734408,2,"POLYGON ((7480169 4235648.142, 7480170.5 42356...",0.000733,0.100310
2,20250902_060832_78_24da_3B_Visual_clip_reproje...,1947.058720,3,"POLYGON ((7479315.351 4233854.092, 7479314.919...",0.001946,0.171176
3,20250902_060832_78_24da_3B_Visual_clip_reproje...,6077.224596,4,"POLYGON ((7478631.278 4237436.841, 7478629.86 ...",0.006074,0.466719
4,20250902_060832_78_24da_3B_Visual_clip_reproje...,170.187945,5,"POLYGON ((7476998.103 4223372.995, 7476996.584...",0.000170,0.048637


## 1. Compute slope (degrees) from DEM

This section computes a **slope raster**. For very large DEMs, you may want to
compute slope externally (e.g. GDAL `gdaldem slope`) and just point `SLOPE_PATH`
to that file.

In [9]:
# one-time reprojection to metres
import rasterio.warp

def reproject_dem_to_area_crs(in_path, out_path, dst_crs=AREA_CRS):
    with rasterio.open(in_path) as src:
        transform, width, height = rasterio.warp.calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        profile = src.profile.copy()
        profile.update({'crs': dst_crs, 'transform': transform,
                        'width': width, 'height': height})
        with rasterio.open(out_path, 'w', **profile) as dst:
            rasterio.warp.reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=rasterio.warp.Resampling.bilinear,
            )

reproject_dem_to_area_crs(DEM_PATH, DEM_PATH_PROJ, dst_crs=AREA_CRS)
# then run compute_slope on the projected DEM (or use gdaldem slope) to rebuild SLOPE_PATH


In [10]:
import math
from rasterio.windows import Window

def compute_slope_blockwise(dem_path, out_path):
    with rasterio.open(dem_path) as src:
        nodata = src.nodata if src.nodata is not None else -9999
        profile = src.profile
        profile.update(dtype='float32', nodata=nodata)
        dx = src.transform.a
        dy = -src.transform.e

        with rasterio.open(out_path, 'w', **profile) as dst:
            for _, win in src.block_windows(1):
                pad = Window(win.col_off-1, win.row_off-1, win.width+2, win.height+2)
                pad = pad.intersection(Window(0, 0, src.width, src.height))
                dem = src.read(1, window=pad, boundless=True, fill_value=nodata).astype('float32')
                mask_arr = dem == nodata

                dzdx = np.zeros_like(dem, dtype='float32')
                dzdy = np.zeros_like(dem, dtype='float32')
                dzdx[:, 1:-1] = (dem[:, 2:] - dem[:, :-2]) / (2 * dx)
                dzdy[1:-1, :] = (dem[2:, :] - dem[:-2, :]) / (2 * dy)

                slope = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))
                slope[mask_arr] = nodata

                r0 = 1 if pad.row_off < win.row_off else 0
                c0 = 1 if pad.col_off < win.col_off else 0
                slope = slope[r0:r0+win.height, c0:c0+win.width]

                dst.write(slope.astype('float32'), 1, window=win)

    print(f'Slope raster written to {out_path}')

compute_slope_blockwise(DEM_PATH_PROJ, SLOPE_PATH)


Slope raster written to D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/outputs/HKH_SRTM_30m_slope_deg.tif


## 2. Helper: mean raster value over a polygon

In [17]:
from rasterio.windows import Window

def mean_raster_over_geom(raster_path, geom, in_crs=AREA_CRS):
    with rasterio.open(raster_path) as src:
        geom_proj = gpd.GeoSeries([geom], crs=in_crs).to_crs(src.crs).iloc[0]
        if not geom_proj.intersects(box(*src.bounds)):
            return np.nan
        win = src.window(*geom_proj.bounds).intersection(Window(0, 0, src.width, src.height))
        if win.width <= 0 or win.height <= 0:
            return np.nan
        try:
            out_img, _ = mask(src, [mapping(geom_proj)], crop=True)
        except ValueError:
            return np.nan
        arr = out_img[0].astype("float32")
        nodata_val = src.nodata if src.nodata is not None else -9999
        arr = arr[(arr != nodata_val) & ~np.isnan(arr)]
        return float(arr.mean()) if arr.size else np.nan


## 3. G11 – Mean slope of parent glacier

In [18]:
from shapely.geometry import Point

gl_sindex = glaciers.sindex

def find_parent_glacier(lake_geom, max_dist=5000):
    """Find parent glacier for a lake.
    1) If any glaciers intersect the lake, pick the one with max intersect area.
    2) Else, pick the nearest glacier within `max_dist` (m).
    Returns glacier index or None.
    """
    possible = list(gl_sindex.intersection(lake_geom.bounds))
    if possible:
        cand = glaciers.iloc[possible]
        inter = cand[cand.intersects(lake_geom)]
        if len(inter) > 0:
            inter_areas = inter.intersection(lake_geom).area
            return inter.loc[inter_areas.idxmax()].name

    lake_cent = lake_geom.centroid
    buf = lake_geom.buffer(max_dist)
    possible = list(gl_sindex.intersection(buf.bounds))
    if not possible:
        return None
    cand = glaciers.iloc[possible].copy()
    cand['dist'] = cand.geometry.centroid.distance(lake_cent)
    cand = cand[cand['dist'] <= max_dist]
    if len(cand) == 0:
        return None
    return cand['dist'].idxmin()

lakes['parent_glacier_idx'] = lakes.geometry.apply(find_parent_glacier)
print('Parent glacier indices assigned.')

Parent glacier indices assigned.


In [19]:
G11_vals = []
for idx, row in lakes.iterrows():
    gl_idx = row['parent_glacier_idx']
    if pd.isna(gl_idx):
        G11_vals.append(np.nan)
        continue
    gl_geom = glaciers.loc[gl_idx, 'geometry']
    mean_slope = mean_raster_over_geom(SLOPE_PATH, gl_geom, in_crs=AREA_CRS)
    G11_vals.append(mean_slope)

lakes['G11_mean_glacier_slope_deg'] = G11_vals
lakes[['lake_id', 'G11_mean_glacier_slope_deg']].head()

,lake_id,G11_mean_glacier_slope_deg
0,1,16.026220
1,2,16.026220
2,3,16.026220
3,4,15.192993
4,5,19.518148


## 4. R1 – Steep non-glacierized area (>30°) around lake

In [ ]:
glaciers_union = unary_union(glaciers.geometry)

def steep_nonglacier_area(lake_geom,
                          buffer_m=R1_BUFFER_M,
                          slope_thresh=R1_SLOPE_THRESH):
    with rasterio.open(SLOPE_PATH) as src:
        lake_proj = gpd.GeoSeries([lake_geom], crs=AREA_CRS).to_crs(src.crs).iloc[0]
        glac_proj = gpd.GeoSeries([glaciers_union], crs=AREA_CRS).to_crs(src.crs).iloc[0]

        lake_buffer = lake_proj.buffer(buffer_m)
        non_glac = lake_buffer.difference(glac_proj)
        if non_glac.is_empty:
            return 0.0

        bounds = non_glac.bounds
        window = src.window(*bounds)
        slope = src.read(1, window=window, masked=True)
        transform = src.window_transform(window)

        mask_geom = geometry_mask([mapping(non_glac)], transform=transform,
                                  invert=True, out_shape=slope.shape)

        data = np.where(mask_geom, slope, np.nan)
        data = np.ma.masked_invalid(data)

        steep = np.where((data > slope_thresh) & (~data.mask), 1, 0)

        res_x = transform.a
        res_y = -transform.e
        cell_area_m2 = res_x * res_y
        area_m2 = steep.sum() * cell_area_m2
        return area_m2 / 1e6

R1_vals = []
for idx, row in lakes.iterrows():
    R1_vals.append(steep_nonglacier_area(row.geometry))
    print(R1_vals)

lakes['R1_steep_nonglac_km2'] = R1_vals
lakes[['lake_id', 'R1_steep_nonglac_km2']].head()

In [20]:
from rasterio.windows import Window

with rasterio.open(SLOPE_PATH) as slope_ds:
    slope_crs = slope_ds.crs
    glaciers_proj = glaciers.to_crs(slope_crs)
    gl_sindex = glaciers_proj.sindex

    def steep_nonglacier_area_fast(lake_geom, buffer_m=R1_BUFFER_M, slope_thresh=R1_SLOPE_THRESH):
        lake_proj = gpd.GeoSeries([lake_geom], crs=AREA_CRS).to_crs(slope_crs).iloc[0]
        lake_buffer = lake_proj.buffer(buffer_m)

        hits = list(gl_sindex.intersection(lake_buffer.bounds))
        if hits:
            glac_local = unary_union(glaciers_proj.iloc[hits].geometry)
            non_glac = lake_buffer.difference(glac_local)
        else:
            non_glac = lake_buffer
        if non_glac.is_empty:
            return 0.0

        window = slope_ds.window(*non_glac.bounds)
        # clip to raster extent
        window = window.intersection(Window(0, 0, slope_ds.width, slope_ds.height))
        # if nothing overlaps, bail out
        if window.width <= 0 or window.height <= 0:
            return 0.0
        slope = slope_ds.read(1, window=window, masked=True)
        transform = slope_ds.window_transform(window)
        if min(slope.shape) == 0:
            return 0.0

        mask_geom = geometry_mask([mapping(non_glac)], transform=transform,
                                  invert=True, out_shape=slope.shape)
        data = np.where(mask_geom, slope, np.nan)
        data = np.ma.masked_invalid(data)
        steep = np.where((data > slope_thresh) & (~data.mask), 1, 0)

        res_x = transform.a
        res_y = -transform.e
        return float(steep.sum() * res_x * res_y / 1e6)

    R1_vals = [steep_nonglacier_area_fast(geom) for geom in lakes.geometry]
    lakes['R1_steep_nonglac_km2'] = R1_vals
    lakes[['lake_id', 'R1_steep_nonglac_km2']].head()

## 5. D3 – Mean slope of moraine dam (approximation)

In [21]:
def dam_zone_mean_slope(lake_geom, search_radius_m=200):
    """Approximate moraine dam zone around lake outlet and compute mean slope.
    1) Find lowest DEM elevation along lake boundary (approx outlet).
    2) Buffer this point by `search_radius_m`.
    3) Compute mean slope in that zone using SLOPE_PATH.
    """
    with rasterio.open(DEM_PATH_PROJ) as dem_src:
        lake_proj = gpd.GeoSeries([lake_geom], crs=AREA_CRS).to_crs(dem_src.crs).iloc[0]
        boundary = lake_proj.boundary

        N = 100
        pts = [boundary.interpolate(t, normalized=True) for t in np.linspace(0, 1, N)]
        coords = [(p.x, p.y) for p in pts]
        z_vals = list(dem_src.sample(coords))
        z_flat = [float(z[0]) for z in z_vals]

    min_idx = int(np.nanargmin(z_flat))
    outlet_pt = pts[min_idx]
    dam_zone = outlet_pt.buffer(search_radius_m)

    dam_zone_area_crs = gpd.GeoSeries([dam_zone], crs=dem_src.crs).to_crs(AREA_CRS).iloc[0]
    mean_slope = mean_raster_over_geom(SLOPE_PATH, dam_zone_area_crs, in_crs=AREA_CRS)
    return mean_slope

D3_vals = []
for idx, row in lakes.iterrows():
    try:
        D3_vals.append(dam_zone_mean_slope(row.geometry))
    except Exception:
        D3_vals.append(np.nan)

lakes['D3_dam_mean_slope_deg'] = D3_vals
lakes[['lake_id', 'D3_dam_mean_slope_deg']].head()

,lake_id,D3_dam_mean_slope_deg
0,1,4.586604
1,2,11.635249
2,3,4.608697
3,4,13.987003
4,5,39.498985


## 6. (Optional) H7 – Watershed area upstream of lake

Advanced, requires `pysheds`. If you skip this section, H7 will be NaN and the
hazard index will still be computed.

In [22]:
from pysheds.grid import Grid
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import from_bounds, Window
from shapely.geometry import box
import geopandas as gpd

# 1) Compute AOI bounds (lakes union + buffer)
aoi = gpd.GeoSeries(lakes.geometry).unary_union.buffer(5000)  # 5 km margin
with rasterio.open(DEM_PATH_PROJ) as src:
    bounds = aoi.bounds  # (west, south, east, north)
    win = from_bounds(*bounds, transform=src.transform)
    win = win.intersection(Window(0, 0, src.width, src.height))  # clamp to raster

    profile = src.profile.copy()
    profile.update({
        'width': int(win.width),
        'height': int(win.height),
        'transform': rasterio.windows.transform(win, src.transform),
    })
    # Optional: downsample to coarser resolution
    scale = 3  # 30 m -> ~90 m
    profile.update({
        'width': profile['width'] // scale,
        'height': profile['height'] // scale,
        'transform': profile['transform'] * profile['transform'].scale(scale, scale)
    })
    
    with rasterio.open(DEM_PATH_PROJ_SUBSET, 'w', **profile) as dst:
        reproject(
            rasterio.band(src, 1),
            rasterio.band(dst, 1),
            src_window=win,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=profile['transform'],
            dst_crs=src.crs,
            resampling=Resampling.bilinear,
        )

# 2) Run pysheds on the reduced DEM
grid = Grid.from_raster(DEM_PATH_PROJ_SUBSET)
dem = grid.read_raster(DEM_PATH_PROJ_SUBSET)
flooded = grid.fill_depressions(dem)
inflated = grid.resolve_flats(flooded)
flowdir = grid.flowdir(inflated)


# now use flowdir in catchment
with rasterio.open(DEM_PATH_PROJ_SUBSET) as src:
    res_x, res_y = src.transform.a, -src.transform.e
cell_area_km2 = (res_x * res_y) / 1e6

def watershed_area_km2(lake_geom):
    with rasterio.open(DEM_PATH_PROJ_SUBSET) as src:
        lake_proj = gpd.GeoSeries([lake_geom], crs=AREA_CRS).to_crs(src.crs).iloc[0]
        boundary = lake_proj.boundary
        pts = [boundary.interpolate(t, normalized=True) for t in np.linspace(0, 1, 50)]
        z_flat = [float(z[0]) for z in src.sample([(p.x, p.y) for p in pts])]
        outlet_pt = pts[int(np.nanargmin(z_flat))]
        row, col = map(int, ~src.transform * (outlet_pt.x, outlet_pt.y))
    catch = grid.catchment(flowdir, x=col, y=row,
                           out_name='catch', recursionlimit=15000)
    print(float((grid.view('catch') == 1).sum() * cell_area_km2))
    return float((grid.view('catch') == 1).sum() * cell_area_km2)

H7_vals = []
for idx, row in lakes.iterrows():
    try:
        H7_vals.append(watershed_area_km2(row.geometry))
    except Exception:
        H7_vals.append(np.nan)
#
lakes['H7_watershed_km2'] = H7_vals
lakes[['lake_id', 'H7_watershed_km2']].head()

C:\Users\gg\AppData\Local\Temp\ipykernel_7604\447977343.py:9: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  aoi = gpd.GeoSeries(lakes.geometry).unary_union.buffer(5000)  # 5 km margin


,lake_id,H7_watershed_km2
0,1,NaN
1,2,NaN
2,3,NaN
3,4,NaN
4,5,NaN


In [24]:
print(None)

None


## 7. L9 – Lake perimeter (km)

In [23]:
lakes['L9_perimeter_km'] = lakes['perimeter_km']
lakes[['lake_id', 'L9_perimeter_km']].head()

,lake_id,L9_perimeter_km
0,1,0.133774
1,2,0.100310
2,3,0.171176
3,4,0.466719
4,5,0.048637


## 8. Normalize factors and compute hazard index

In [24]:
def normalize_series(s: pd.Series):
    s = s.replace([np.inf, -np.inf], np.nan)
    s_valid = s.dropna()
    if s_valid.empty:
        return pd.Series(np.nan, index=s.index)
    s_min = s_valid.min()
    s_max = s_valid.max()
    if s_max == s_min:
        return pd.Series(0.5, index=s.index)
    return (s - s_min) / (s_max - s_min)

lakes['G11'] = lakes['G11_mean_glacier_slope_deg']
lakes['R1']  = lakes['R1_steep_nonglac_km2']
lakes['D3']  = lakes['D3_dam_mean_slope_deg']
lakes['H7']  = lakes['H7_watershed_km2'] if 'H7_watershed_km2' in lakes.columns else np.nan
lakes['L9']  = lakes['L9_perimeter_km']

for fac in ['G11', 'R1', 'D3', 'H7', 'L9']:
    lakes[f'{fac}_norm'] = normalize_series(lakes[fac])

w_G11 = 0.27
w_R1  = 0.25
w_D3  = 0.21
w_H7  = 0.17
w_L9  = 0.10

lakes['hazard_index'] = (
    w_G11 * lakes['G11_norm'].fillna(0) +
    w_R1  * lakes['R1_norm'].fillna(0)  +
    w_D3  * lakes['D3_norm'].fillna(0)  +
    w_H7  * lakes['H7_norm'].fillna(0)  +
    w_L9  * lakes['L9_norm']
)

lakes[['lake_id', 'hazard_index']].head()

,lake_id,hazard_index
0,1,0.073668
1,2,0.089763
2,3,0.074134
3,4,0.095469
4,5,0.170376


## 9. Classify hazard index (Jenks) and flag PDGLs

In [25]:
hi_valid = lakes['hazard_index'].dropna()
if hi_valid.empty:
    raise ValueError('No valid hazard_index values. Check factor computations.')

nb = mapclassify.NaturalBreaks(hi_valid, k=5)
labels = ['very_low', 'low', 'medium', 'high', 'very_high']

class_indices = nb.yb
class_map = dict(zip(hi_valid.index, class_indices))

def label_for_idx(idx):
    if idx not in class_map:
        return None
    ci = class_map[idx]
    if ci < 0:
        return None
    return labels[ci]

lakes['hazard_class'] = [label_for_idx(idx) for idx in lakes.index]
lakes['hazard_class'] = pd.Categorical(lakes['hazard_class'], categories=labels, ordered=True)
lakes['is_PDGL'] = lakes['hazard_class'].isin(['high', 'very_high'])

print('Hazard class counts:')
print(lakes['hazard_class'].value_counts())
print('\nNumber of PDGLs:', lakes['is_PDGL'].sum())

lakes[['lake_id', 'hazard_index', 'hazard_class', 'is_PDGL']].head()

Hazard class counts:
hazard_class
medium       4638
low          4101
high         3040
very_low     2095
very_high    1479
Name: count, dtype: int64

Number of PDGLs: 4519


,lake_id,hazard_index,hazard_class,is_PDGL
0,1,0.073668,very_low,False
1,2,0.089763,very_low,False
2,3,0.074134,very_low,False
3,4,0.095469,very_low,False
4,5,0.170376,low,False


## 10. Save PDGL-annotated lakes

In [26]:
lakes.to_file(OUT_PDGL_PATH, driver='GPKG')
print(f'PDGL-annotated lakes saved to {OUT_PDGL_PATH}')

PDGL-annotated lakes saved to lakes_2025_PDGL.gpkg
